In [ ]:
import re
from datetime import datetime
import pandas as pd
from dateutil import parser as dateparser

LINE_RE = re.compile(r"^\[(.+?)\] ([^:]+): (.*)$")

def parse_export(path):
    messages = []
    current = None
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = LINE_RE.match(line)
            if m:
                if current:
                    messages.append(current)
                ts, sender, body = m.groups()
                current = {
                    "timestamp": dateparser.parse(ts, fuzzy=True),
                    "sender": sender.strip(),
                    "body": body.rstrip("\n"),
                }
            elif current:
                current["body"] += "\n" + line.rstrip("\n")
    if current:
        messages.append(current)
    df = pd.DataFrame(messages)
    df["is_media"] = df["body"].str.contains("<Media omitted>", na=False)
    df["is_system"] = df["sender"] == "System"
    return df

In [3]:
df = parse_export("sample_chat.txt")
print(df.head())

            timestamp      sender                                    body  \
0 2026-03-15 14:23:01    John Doe                   Welcome to the group!   
1 2026-03-15 14:24:15  Jane Smith                         <Media omitted>   
2 2026-03-15 14:25:30    John Doe  When is the next meeting we agreed on?   
3 2026-03-15 14:26:02      System               John Doe added Bob Wilson   
4 2026-03-15 14:27:10  Bob Wilson       Hi everyone, thanks for adding me   

   is_media  is_system  
0     False      False  
1      True      False  
2     False      False  
3     False       True  
4     False      False  


First we cover the easy anaytics:
1. Hour frequency
2. User frequency
3. Daily and weekly frequencies

In [15]:
# create the summary 
def sentiment_analysis(text):
    # Placeholder for sentiment analysis logic
    return 0  # This is just a placeholder value

def compute_summary(df):
    real = df[~df['is_system']]
    return {
        "total_messages": int(len(real)),
        "unique_senders": int(real["sender"].nunique()),
        "media_message_count": int(real['is_media'].sum()),
        "date_range": {
            "first": real["timestamp"].min().date(),
            "last": real["timestamp"].max().date()
        },       
        "text_message_count": int((~real['is_media']).sum())
    }
# frequency by hour
def compute_peak_hours(df):
    counts = df[~df["is_system"]]["timestamp"].dt.hour.value_counts().reindex(range(24), fill_value=0)
    return [{"hour": int(h), "count": int(c)} for h, c in counts.items()]

#compute the summary for each sender, for sentiment use the mean value
def compute_per_user(df):
    real= df[~df["is_system"]]
    real["sentiment"] = real["body"].apply(sentiment_analysis) # placeholder for sentiment analysis function
    return (
        real
        .groupby("sender")
        .agg(
            messages=("body", "count"),
            media=("is_media", "sum"),
            avg_sentiment=("sentiment", "mean")
        )
        .sort_values("messages", ascending=False)
        )

def compute_sentiment(df):
    # Placeholder for sentiment analysis logic for the chat
    return 0  # This is just a placeholder value

def compute_top_influencers(df):
    #placeholder for influencer analysis logic, we can use message count as a simple proxy for influence
    return df[~df["is_system"]]["sender"].value_counts().head(5).index.tolist()


def compute_frequency(df):
    real = df[~df["is_system"]]
    iso_date = real["timestamp"].dt.isocalendar()
    real = real.copy()
    real["year_week"] = iso_date["year"].astype(str) + "-W" + iso_date["week"].astype(str)

    daily = real.groupby(real["timestamp"].dt.date).size().rename("count")
    weekly = real.groupby(real["year_week"]).size().rename("count")

    return {
        "daily": daily,
        "weekly": weekly,
    }

print (compute_frequency(df))

#print (compute_per_user(df))

# frequency by sender (per user)
print (df.groupby(df["sender"]).size().sort_values(ascending=False))

# frequency by day and week (per date)
print (df.groupby(df["timestamp"].dt.date).size())
print (df.groupby(df["timestamp"].dt.isocalendar().week).size())

# summary (System sender included)
summary = compute_summary(df)
print(summary["total_messages"]) #total message count
print(summary["text_message_count"]) #text message count
print(summary["unique_senders"]) #unique sender count
print(summary["media_message_count"]) #media message count
print(summary["date_range"]["first"]) #first message date
print(summary["date_range"]["last"]) #last message date


{'daily': timestamp
2026-03-15    60
2026-03-16    30
Name: count, dtype: int64, 'weekly': year_week
2026-W11    60
2026-W12    30
Name: count, dtype: int64}
sender
Jane Smith      24
John Doe        19
Mike Brown      17
Bob Wilson      15
Sarah Connor    12
System           9
dtype: int64
timestamp
2026-03-15    64
2026-03-16    32
dtype: int64
week
11    64
12    32
dtype: int64
90
89
6
1
2026-03-15
2026-03-16
